In [1]:
# ============================================================
# CELL 1 — LOAD REQUIRED TABLES
# ============================================================

import pandas as pd
import numpy as np

# Load dimensions
dim_date = pd.read_csv("../data/raw/dim_date.csv")
dim_doctor = pd.read_csv("../data/raw/dim_doctor.csv")
dim_department = pd.read_csv("../data/raw/dim_department.csv")
dim_diagnosis = pd.read_csv("../data/raw/dim_diagnosis.csv")

# Load completed appointments
fact_appointments = pd.read_csv(
    "../data/raw/fact_appointments.csv"
)

# Convert date/time columns
dim_date["full_date"] = pd.to_datetime(
    dim_date["full_date"]
)

fact_appointments["scheduled_datetime"] = pd.to_datetime(
    fact_appointments["scheduled_datetime"]
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("Tables loaded successfully!")
print()

print("dim_date:", dim_date.shape)
print("dim_doctor:", dim_doctor.shape)
print("dim_department:", dim_department.shape)
print("dim_diagnosis:", dim_diagnosis.shape)
print("fact_appointments:", fact_appointments.shape)

print("\nCompleted appointments:")
print(
    (fact_appointments["status"] == "Completed").sum()
)

print("\nEmergency encounters will be added separately later.")

Tables loaded successfully!

dim_date: (731, 11)
dim_doctor: (600, 9)
dim_department: (44, 5)
dim_diagnosis: (12, 4)
fact_appointments: (500000, 14)

Completed appointments:
415965

Emergency encounters will be added separately later.


In [2]:
# ============================================================
# CELL 2 — SELECT APPOINTMENTS FOR ENCOUNTERS
# ============================================================

# Only completed appointments are eligible
completed_appointments = fact_appointments[
    fact_appointments["status"] == "Completed"
].copy()

# Target encounter population
N_ENCOUNTERS = 300_000

# Randomly select completed appointments
selected_appointments = completed_appointments.sample(
    n=N_ENCOUNTERS,
    random_state=42
).reset_index(drop=True)

print("Encounter appointment pool created!")
print()
print("Completed appointments available:",
      len(completed_appointments))

print("Target encounters:",
      N_ENCOUNTERS)

print("Selected appointments:",
      len(selected_appointments))

print(
    "\nDuplicate appointment IDs:",
    selected_appointments["appointment_id"].duplicated().sum()
)

print("\nSample:")
display(
    selected_appointments[
        [
            "appointment_id",
            "patient_id",
            "doctor_id",
            "department_id",
            "hospital_id",
            "appointment_type",
            "scheduled_datetime"
        ]
    ].head(10)
)

Encounter appointment pool created!

Completed appointments available: 415965
Target encounters: 300000
Selected appointments: 300000

Duplicate appointment IDs: 0

Sample:


,appointment_id,patient_id,doctor_id,department_id,hospital_id,appointment_type,scheduled_datetime
0,A211909,P068956,DR0362,D004,H001,Consultation,2025-02-20 08:06:00
1,A261505,P082895,DR0151,D011,H002,Consultation,2025-01-04 14:33:00
2,A071055,P002135,DR0130,D034,H006,Follow-up,2025-08-16 12:40:00
3,A219226,P029583,DR0366,D012,H002,Diagnostic,2025-06-08 09:28:00
4,A374155,P018029,DR0420,D014,H003,Consultation,2025-06-06 15:21:00
5,A458515,P067837,DR0263,D008,H002,Consultation,2025-12-17 13:56:00
6,A485974,P007889,DR0234,D041,H008,Follow-up,2025-02-26 09:51:00
7,A033963,P061983,DR0591,D028,H005,Consultation,2024-01-23 15:26:00
8,A296035,P079281,DR0468,D008,H002,Consultation,2025-09-28 14:06:00
9,A444194,P025020,DR0393,D029,H005,Consultation,2025-08-04 17:13:00


In [3]:
# ============================================================
# CELL 3 — CREATE ENCOUNTER IDs & LINK APPOINTMENTS
# ============================================================

# Create unique encounter IDs
selected_appointments["encounter_id"] = [
    f"E{i:06d}"
    for i in range(1, N_ENCOUNTERS + 1)
]

# Create the initial encounter table
fact_encounters = selected_appointments[
    [
        "encounter_id",
        "appointment_id",
        "patient_id",
        "doctor_id",
        "department_id",
        "hospital_id",
        "appointment_date_id",
        "scheduled_datetime"
    ]
].copy()

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Encounter IDs and appointment links created!")

print(
    "\nTotal encounters:",
    len(fact_encounters)
)

print(
    "Unique encounter IDs:",
    fact_encounters["encounter_id"].nunique()
)

print(
    "Duplicate encounter IDs:",
    fact_encounters["encounter_id"].duplicated().sum()
)

print(
    "Missing appointment IDs:",
    fact_encounters["appointment_id"].isna().sum()
)

print(
    "Missing patient IDs:",
    fact_encounters["patient_id"].isna().sum()
)

print("\nSample:")

display(
    fact_encounters.head(10)
)

Encounter IDs and appointment links created!

Total encounters: 300000
Unique encounter IDs: 300000
Duplicate encounter IDs: 0
Missing appointment IDs: 0
Missing patient IDs: 0

Sample:


,encounter_id,appointment_id,patient_id,doctor_id,department_id,hospital_id,appointment_date_id,scheduled_datetime
0,E000001,A211909,P068956,DR0362,D004,H001,20250220,2025-02-20 08:06:00
1,E000002,A261505,P082895,DR0151,D011,H002,20250104,2025-01-04 14:33:00
2,E000003,A071055,P002135,DR0130,D034,H006,20250816,2025-08-16 12:40:00
3,E000004,A219226,P029583,DR0366,D012,H002,20250608,2025-06-08 09:28:00
4,E000005,A374155,P018029,DR0420,D014,H003,20250606,2025-06-06 15:21:00
5,E000006,A458515,P067837,DR0263,D008,H002,20251217,2025-12-17 13:56:00
6,E000007,A485974,P007889,DR0234,D041,H008,20250226,2025-02-26 09:51:00
7,E000008,A033963,P061983,DR0591,D028,H005,20240123,2024-01-23 15:26:00
8,E000009,A296035,P079281,DR0468,D008,H002,20250928,2025-09-28 14:06:00
9,E000010,A444194,P025020,DR0393,D029,H005,20250804,2025-08-04 17:13:00


In [4]:
# ============================================================
# CELL 4 — ASSIGN ENCOUNTER TYPE
# ============================================================

# Generate encounter types
fact_encounters["encounter_type"] = np.random.choice(
    [
        "Outpatient",
        "Inpatient",
        "Emergency"
    ],
    size=len(fact_encounters),
    p=[
        0.72,   # Outpatient
        0.18,   # Inpatient
        0.10    # Emergency
    ]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Encounter types assigned successfully!")

print("\nEncounter type distribution:")

display(
    fact_encounters["encounter_type"]
    .value_counts()
)

print("\nEncounter type percentage:")

display(
    (
        fact_encounters["encounter_type"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
)

Encounter types assigned successfully!

Encounter type distribution:


encounter_type
Outpatient    215790
Inpatient      54184
Emergency      30026
Name: count, dtype: int64


Encounter type percentage:


encounter_type
Outpatient    71.93
Inpatient     18.06
Emergency     10.01
Name: proportion, dtype: float64

In [5]:
# ============================================================
# CELL 5 — ASSIGN DIAGNOSIS
# ============================================================

diagnosis_ids = dim_diagnosis["diagnosis_id"].tolist()

# Weighted diagnosis distribution
diagnosis_probabilities = [
    0.12,  # DG001 Hypertension
    0.12,  # DG002 Type 2 Diabetes
    0.11,  # DG003 Acute Respiratory Infection
    0.09,  # DG004 Back Pain
    0.08,  # DG005 GERD
    0.07,  # DG006 Ischemic Heart Disease
    0.08,  # DG007 Migraine
    0.08,  # DG008 Fracture
    0.07,  # DG009 Breast Cancer
    0.06,  # DG010 Lung Cancer
    0.06,  # DG011 Dermatitis
    0.06   # DG012 Urinary Infection
]

fact_encounters["diagnosis_id"] = np.random.choice(
    diagnosis_ids,
    size=len(fact_encounters),
    p=diagnosis_probabilities
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Diagnosis assignment completed!")

print(
    "\nMissing diagnosis IDs:",
    fact_encounters["diagnosis_id"].isna().sum()
)

print(
    "Unique diagnosis categories:",
    fact_encounters["diagnosis_id"].nunique()
)

print("\nDiagnosis distribution:")

display(
    fact_encounters["diagnosis_id"]
    .value_counts()
    .sort_index()
)

Diagnosis assignment completed!

Missing diagnosis IDs: 0
Unique diagnosis categories: 12

Diagnosis distribution:


diagnosis_id
DG001    35886
DG002    35802
DG003    32978
DG004    26933
DG005    24189
DG006    21063
DG007    23810
DG008    24231
DG009    20905
DG010    17863
DG011    18065
DG012    18275
Name: count, dtype: int64

In [6]:
# ============================================================
# CELL 6 — GENERATE ENCOUNTER TIMESTAMPS
# ============================================================

# Convert scheduled datetime to datetime
fact_encounters["scheduled_datetime"] = pd.to_datetime(
    fact_encounters["scheduled_datetime"]
)

# ------------------------------------------------------------
# Arrival time
# ------------------------------------------------------------

# Patients arrive around their scheduled time
arrival_offset_minutes = np.random.normal(
    loc=5,
    scale=20,
    size=len(fact_encounters)
)

arrival_offset_minutes = np.clip(
    arrival_offset_minutes,
    -30,
    90
)

fact_encounters["arrival_datetime"] = (
    fact_encounters["scheduled_datetime"]
    + pd.to_timedelta(arrival_offset_minutes, unit="m")
)

# ------------------------------------------------------------
# Consultation start time
# ------------------------------------------------------------

# Waiting time will be added in the next step,
# so initially create the column from arrival time.

fact_encounters["consultation_start_datetime"] = (
    fact_encounters["arrival_datetime"]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Encounter timestamp base created!")

print(
    "\nMissing arrival timestamps:",
    fact_encounters["arrival_datetime"].isna().sum()
)

print(
    "Missing consultation start timestamps:",
    fact_encounters["consultation_start_datetime"].isna().sum()
)

print("\nArrival datetime range:")
print(
    fact_encounters["arrival_datetime"].min(),
    "to",
    fact_encounters["arrival_datetime"].max()
)

Encounter timestamp base created!

Missing arrival timestamps: 0
Missing consultation start timestamps: 0

Arrival datetime range:
2024-01-02 10:23:51.161023818 to 2025-12-31 20:41:15.116688120


In [7]:
# ============================================================
# CELL 7 — GENERATE WAIT TIME
# ============================================================

# Base waiting time by encounter type
base_wait = fact_encounters["encounter_type"].map({
    "Outpatient": 28,
    "Inpatient": 45,
    "Emergency": 18
})

# Department-level operational pressure
department_capacity = dim_department.set_index(
    "department_id"
)["capacity_per_day"]

fact_encounters["department_capacity"] = (
    fact_encounters["department_id"]
    .map(department_capacity)
)

# Smaller capacity departments tend to experience slightly
# higher waiting times under the same patient load.
capacity_factor = (
    40 / fact_encounters["department_capacity"]
)

# Random operational variation
random_wait = np.random.gamma(
    shape=2.2,
    scale=10,
    size=len(fact_encounters)
)

# Final waiting time
wait_time = (
    base_wait
    + (capacity_factor * 15)
    + random_wait
)

# Add a small number of legitimate high-wait cases
high_wait_mask = np.random.random(len(fact_encounters)) < 0.015

wait_time[high_wait_mask] *= np.random.uniform(
    2.0,
    4.0,
    high_wait_mask.sum()
)

# Keep values realistic
wait_time = np.clip(
    wait_time,
    5,
    240
)

fact_encounters["wait_time_minutes"] = np.round(
    wait_time
).astype(int)

# Remove temporary column
fact_encounters.drop(
    columns=["department_capacity"],
    inplace=True
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Wait time generation completed!")

print(
    "\nMissing wait times:",
    fact_encounters["wait_time_minutes"].isna().sum()
)

print(
    "Minimum wait time:",
    fact_encounters["wait_time_minutes"].min(),
    "minutes"
)

print(
    "Maximum wait time:",
    fact_encounters["wait_time_minutes"].max(),
    "minutes"
)

print(
    "Average wait time:",
    round(
        fact_encounters["wait_time_minutes"].mean(),
        2
    ),
    "minutes"
)

print(
    "Median wait time:",
    fact_encounters["wait_time_minutes"].median(),
    "minutes"
)

Wait time generation completed!

Missing wait times: 0
Minimum wait time: 23 minutes
Maximum wait time: 240 minutes
Average wait time: 62.18 minutes
Median wait time: 58.0 minutes


In [8]:
# ============================================================
# CELL 8 — GENERATE CONSULTATION TIME
# ============================================================

# Base consultation time by encounter type
base_consultation = fact_encounters["encounter_type"].map({
    "Outpatient": 18,
    "Inpatient": 35,
    "Emergency": 25
})

# Diagnosis-related complexity
diagnosis_complexity = fact_encounters["diagnosis_id"].map({
    "DG001": 1.00,  # Hypertension
    "DG002": 1.05,  # Diabetes
    "DG003": 0.90,  # Respiratory infection
    "DG004": 0.85,  # Back pain
    "DG005": 0.90,  # GERD
    "DG006": 1.35,  # Ischemic heart disease
    "DG007": 0.95,  # Migraine
    "DG008": 1.15,  # Fracture
    "DG009": 1.45,  # Breast cancer
    "DG010": 1.50,  # Lung cancer
    "DG011": 0.85,  # Dermatitis
    "DG012": 0.90   # Urinary infection
})

# Random variation
random_consultation = np.random.gamma(
    shape=2.5,
    scale=5,
    size=len(fact_encounters)
)

# Final consultation duration
consultation_time = (
    base_consultation
    * diagnosis_complexity
    + random_consultation
)

# Keep within realistic operational limits
consultation_time = np.clip(
    consultation_time,
    8,
    120
)

fact_encounters["consultation_time_minutes"] = np.round(
    consultation_time
).astype(int)

# ------------------------------------------------------------
# Create actual consultation start time
# ------------------------------------------------------------

fact_encounters["consultation_start_datetime"] = (
    fact_encounters["arrival_datetime"]
    + pd.to_timedelta(
        fact_encounters["wait_time_minutes"],
        unit="m"
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Consultation time generation completed!")

print(
    "\nMissing consultation times:",
    fact_encounters["consultation_time_minutes"].isna().sum()
)

print(
    "Minimum consultation time:",
    fact_encounters["consultation_time_minutes"].min(),
    "minutes"
)

print(
    "Maximum consultation time:",
    fact_encounters["consultation_time_minutes"].max(),
    "minutes"
)

print(
    "Average consultation time:",
    round(
        fact_encounters["consultation_time_minutes"].mean(),
        2
    ),
    "minutes"
)

print(
    "Median consultation time:",
    fact_encounters["consultation_time_minutes"].median(),
    "minutes"
)

Consultation time generation completed!

Missing consultation times: 0
Minimum consultation time: 15 minutes
Maximum consultation time: 120 minutes
Average consultation time: 35.4 minutes
Median consultation time: 33.0 minutes


In [9]:
# ============================================================
# CELL 9 — GENERATE DISCHARGE TIMESTAMP
# ============================================================

# Base post-consultation duration by encounter type
post_consultation_duration = fact_encounters["encounter_type"].map({
    "Outpatient": 10,
    "Inpatient": 180,
    "Emergency": 60
})

# Random variation in post-consultation duration
random_duration = np.random.gamma(
    shape=2.0,
    scale=8,
    size=len(fact_encounters)
)

# Total duration after consultation starts
total_post_consultation = (
    post_consultation_duration
    + random_duration
)

# Keep operational duration realistic
total_post_consultation = np.clip(
    total_post_consultation,
    5,
    1440
)

# ------------------------------------------------------------
# Generate discharge timestamp
# ------------------------------------------------------------

fact_encounters["discharge_datetime"] = (
    fact_encounters["consultation_start_datetime"]
    + pd.to_timedelta(
        total_post_consultation,
        unit="m"
    )
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Discharge timestamp generation completed!")

print(
    "\nMissing discharge timestamps:",
    fact_encounters["discharge_datetime"].isna().sum()
)

print("\nDischarge datetime range:")
print(
    fact_encounters["discharge_datetime"].min(),
    "to",
    fact_encounters["discharge_datetime"].max()
)

# Timeline validation
invalid_timeline = (
    (fact_encounters["arrival_datetime"] >=
     fact_encounters["consultation_start_datetime"])
    |
    (fact_encounters["consultation_start_datetime"] >=
     fact_encounters["discharge_datetime"])
).sum()

print(
    "\nInvalid encounter timelines:",
    invalid_timeline
)

Discharge timestamp generation completed!

Missing discharge timestamps: 0

Discharge datetime range:
2024-01-02 12:02:39.970918998 to 2026-01-01 01:27:39.806044776

Invalid encounter timelines: 0


In [10]:
# ============================================================
# CELL 10 — ADMISSION & DISCHARGE STATUS
# ============================================================

# Admission probability by encounter type
admission_probability = fact_encounters["encounter_type"].map({
    "Outpatient": 0.03,
    "Inpatient": 0.92,
    "Emergency": 0.35
})

# Generate admission flag
fact_encounters["admission_flag"] = (
    np.random.random(len(fact_encounters))
    < admission_probability
).astype(int)

# ------------------------------------------------------------
# Generate discharge status
# ------------------------------------------------------------

random_status = np.random.random(len(fact_encounters))

fact_encounters["discharge_status"] = np.select(
    [
        random_status < 0.93,
        random_status < 0.97
    ],
    [
        "Discharged",
        "Referred"
    ],
    default="Transferred"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Admission & discharge status generation completed!")

print("\nAdmission flag distribution:")
print(
    fact_encounters["admission_flag"]
    .value_counts()
    .sort_index()
)

print("\nDischarge status distribution:")
print(
    fact_encounters["discharge_status"]
    .value_counts()
)

print(
    "\nMissing admission flags:",
    fact_encounters["admission_flag"].isna().sum()
)

print(
    "Missing discharge statuses:",
    fact_encounters["discharge_status"].isna().sum()
)

Admission & discharge status generation completed!

Admission flag distribution:
admission_flag
0    233100
1     66900
Name: count, dtype: int64

Discharge status distribution:
discharge_status
Discharged     278946
Referred        11964
Transferred      9090
Name: count, dtype: int64

Missing admission flags: 0
Missing discharge statuses: 0


In [11]:
# ============================================================
# CELL 11 — CALCULATE 30-DAY READMISSION
# ============================================================

# Work with a copy sorted by patient and encounter date
readmission_check = fact_encounters[
    [
        "encounter_id",
        "patient_id",
        "arrival_datetime",
        "discharge_datetime",
        "admission_flag"
    ]
].copy()

readmission_check = readmission_check.sort_values(
    ["patient_id", "arrival_datetime"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Find the next encounter for each patient
# ------------------------------------------------------------

readmission_check["next_arrival_datetime"] = (
    readmission_check
    .groupby("patient_id")["arrival_datetime"]
    .shift(-1)
)

readmission_check["next_admission_flag"] = (
    readmission_check
    .groupby("patient_id")["admission_flag"]
    .shift(-1)
)

# Days between discharge and next encounter
readmission_check["days_to_next_encounter"] = (
    (
        readmission_check["next_arrival_datetime"]
        - readmission_check["discharge_datetime"]
    ).dt.total_seconds()
    / (24 * 60 * 60)
)

# ------------------------------------------------------------
# Readmission rule
# ------------------------------------------------------------
# A readmission occurs when:
# 1. Current encounter was an admission
# 2. Patient has a subsequent admission
# 3. Subsequent admission occurs within 30 days after discharge

readmission_check["readmission_30d_flag"] = (
    (readmission_check["admission_flag"] == 1)
    &
    (readmission_check["next_admission_flag"] == 1)
    &
    (readmission_check["days_to_next_encounter"] > 0)
    &
    (readmission_check["days_to_next_encounter"] <= 30)
).astype(int)

# ------------------------------------------------------------
# Map result back to fact table
# ------------------------------------------------------------

readmission_flags = readmission_check[
    ["encounter_id", "readmission_30d_flag"]
]

fact_encounters = fact_encounters.merge(
    readmission_flags,
    on="encounter_id",
    how="left"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("30-day readmission calculation completed!")

print(
    "\nMissing readmission flags:",
    fact_encounters["readmission_30d_flag"].isna().sum()
)

print("\nReadmission distribution:")
print(
    fact_encounters["readmission_30d_flag"]
    .value_counts()
    .sort_index()
)

# Calculate rate among admitted encounters
admitted_encounters = (
    fact_encounters["admission_flag"] == 1
)

readmissions = (
    fact_encounters.loc[
        admitted_encounters,
        "readmission_30d_flag"
    ].sum()
)

admission_count = admitted_encounters.sum()

readmission_rate = (
    readmissions / admission_count * 100
)

print(
    "\n30-day readmissions:",
    int(readmissions)
)

print(
    "Admitted encounters:",
    int(admission_count)
)

print(
    "30-day readmission rate:",
    round(readmission_rate, 2),
    "%"
)


30-day readmission calculation completed!

Missing readmission flags: 0

Readmission distribution:
readmission_30d_flag
0    297560
1      2440
Name: count, dtype: int64

30-day readmissions: 2440
Admitted encounters: 66900
30-day readmission rate: 3.65 %


In [12]:
# ============================================================
# CELL 12 — FINAL ENCOUNTER VALIDATION
# ============================================================

required_columns = [
    "encounter_id",
    "appointment_id",
    "patient_id",
    "doctor_id",
    "department_id",
    "hospital_id",
    "appointment_date_id",
    "scheduled_datetime",
    "encounter_type",
    "arrival_datetime",
    "consultation_start_datetime",
    "discharge_datetime",
    "wait_time_minutes",
    "consultation_time_minutes",
    "admission_flag",
    "discharge_status",
    "diagnosis_id",
    "readmission_30d_flag"
]

print("Rows:", len(fact_encounters))
print("Columns:", len(fact_encounters.columns))

print("\nMissing values:")
display(
    fact_encounters[required_columns]
    .isna()
    .sum()
)

print("\nDuplicate encounter IDs:",
      fact_encounters["encounter_id"].duplicated().sum())

print("\nDuplicate appointment IDs:",
      fact_encounters["appointment_id"].duplicated().sum())

# ------------------------------------------------------------
# Timeline validation
# ------------------------------------------------------------

invalid_timeline = (
    (fact_encounters["arrival_datetime"] >=
     fact_encounters["consultation_start_datetime"])
    |
    (fact_encounters["consultation_start_datetime"] >=
     fact_encounters["discharge_datetime"])
).sum()

print("\nInvalid timestamp sequences:", invalid_timeline)

# ------------------------------------------------------------
# Appointment → encounter consistency
# ------------------------------------------------------------

appointment_mismatch = (
    fact_encounters["appointment_id"].notna()
    &
    (
        fact_encounters["appointment_id"]
        .duplicated(keep=False)
    )
)

print(
    "Appointments linked to multiple encounters:",
    appointment_mismatch.sum()
)

# ------------------------------------------------------------
# Key distributions
# ------------------------------------------------------------

print("\nEncounter type:")
print(fact_encounters["encounter_type"].value_counts())

print("\nAdmission flag:")
print(fact_encounters["admission_flag"].value_counts())

print("\nDischarge status:")
print(fact_encounters["discharge_status"].value_counts())

print("\nReadmission flag:")
print(fact_encounters["readmission_30d_flag"].value_counts())

Rows: 300000
Columns: 18

Missing values:


encounter_id                   0
appointment_id                 0
patient_id                     0
doctor_id                      0
department_id                  0
hospital_id                    0
appointment_date_id            0
scheduled_datetime             0
encounter_type                 0
arrival_datetime               0
consultation_start_datetime    0
discharge_datetime             0
wait_time_minutes              0
consultation_time_minutes      0
admission_flag                 0
discharge_status               0
diagnosis_id                   0
readmission_30d_flag           0
dtype: int64


Duplicate encounter IDs: 0

Duplicate appointment IDs: 0

Invalid timestamp sequences: 0
Appointments linked to multiple encounters: 0

Encounter type:
encounter_type
Outpatient    215790
Inpatient      54184
Emergency      30026
Name: count, dtype: int64

Admission flag:
admission_flag
0    233100
1     66900
Name: count, dtype: int64

Discharge status:
discharge_status
Discharged     278946
Referred        11964
Transferred      9090
Name: count, dtype: int64

Readmission flag:
readmission_30d_flag
0    297560
1      2440
Name: count, dtype: int64


In [13]:
# ============================================================
# CELL 13 — SAVE FACT ENCOUNTERS
# ============================================================

output_path = "../data/raw/fact_encounters.csv"

fact_encounters.to_csv(
    output_path,
    index=False
)

print("Fact encounters saved successfully!")

print(
    "\nFile path:",
    output_path
)

print(
    "Rows:",
    len(fact_encounters)
)

print(
    "Columns:",
    len(fact_encounters.columns)
)

Fact encounters saved successfully!

File path: ../data/raw/fact_encounters.csv
Rows: 300000
Columns: 18


In [14]:
# ============================================================
# CELL 14 — RELOAD & VALIDATE FACT ENCOUNTERS
# ============================================================

encounters_path = "../data/raw/fact_encounters.csv"

fact_encounters_check = pd.read_csv(
    encounters_path,
    parse_dates=[
        "scheduled_datetime",
        "arrival_datetime",
        "consultation_start_datetime",
        "discharge_datetime"
    ]
)

print("Fact encounters reloaded successfully!")

print("\nRows:", len(fact_encounters_check))
print("Columns:", len(fact_encounters_check.columns))

print(
    "\nDuplicate encounter IDs:",
    fact_encounters_check["encounter_id"].duplicated().sum()
)

print(
    "Duplicate appointment IDs:",
    fact_encounters_check["appointment_id"].duplicated().sum()
)

print(
    "Missing encounter IDs:",
    fact_encounters_check["encounter_id"].isna().sum()
)

print(
    "Missing patient IDs:",
    fact_encounters_check["patient_id"].isna().sum()
)

print(
    "Missing diagnosis IDs:",
    fact_encounters_check["diagnosis_id"].isna().sum()
)

print(
    "Missing discharge timestamps:",
    fact_encounters_check["discharge_datetime"].isna().sum()
)

print("\nFinal columns:")
print(fact_encounters_check.columns.tolist())

Fact encounters reloaded successfully!

Rows: 300000
Columns: 18

Duplicate encounter IDs: 0
Duplicate appointment IDs: 0
Missing encounter IDs: 0
Missing patient IDs: 0
Missing diagnosis IDs: 0
Missing discharge timestamps: 0

Final columns:
['encounter_id', 'appointment_id', 'patient_id', 'doctor_id', 'department_id', 'hospital_id', 'appointment_date_id', 'scheduled_datetime', 'encounter_type', 'diagnosis_id', 'arrival_datetime', 'consultation_start_datetime', 'wait_time_minutes', 'consultation_time_minutes', 'discharge_datetime', 'admission_flag', 'discharge_status', 'readmission_30d_flag']


In [15]:
# ============================================================
# CELL 15 — LOAD BILLING DEPENDENCIES
# ============================================================

# Load required dimension and fact tables

dim_hospital = pd.read_csv(
    "../data/raw/dim_hospital.csv"
)

dim_department = pd.read_csv(
    "../data/raw/dim_department.csv"
)

dim_insurance = pd.read_csv(
    "../data/raw/dim_insurance.csv"
)

dim_diagnosis = pd.read_csv(
    "../data/raw/dim_diagnosis.csv"
)

fact_encounters = pd.read_csv(
    "../data/raw/fact_encounters.csv",
    parse_dates=[
        "scheduled_datetime",
        "arrival_datetime",
        "consultation_start_datetime",
        "discharge_datetime"
    ]
)

# ------------------------------------------------------------
# Basic validation
# ------------------------------------------------------------

print("Billing dependencies loaded successfully!")

print("\nTable shapes:")

print("dim_hospital:", dim_hospital.shape)
print("dim_department:", dim_department.shape)
print("dim_insurance:", dim_insurance.shape)
print("dim_diagnosis:", dim_diagnosis.shape)
print("fact_encounters:", fact_encounters.shape)

print(
    "\nUnique encounters:",
    fact_encounters["encounter_id"].nunique()
)

print(
    "Unique patients:",
    fact_encounters["patient_id"].nunique()
)

Billing dependencies loaded successfully!

Table shapes:
dim_hospital: (8, 7)
dim_department: (44, 5)
dim_insurance: (8, 6)
dim_diagnosis: (12, 4)
fact_encounters: (300000, 18)

Unique encounters: 300000
Unique patients: 95004


In [16]:
# ============================================================
# CELL 16 — CREATE BILLING BASE TABLE
# ============================================================

N_BILLS = len(fact_encounters)

# ------------------------------------------------------------
# Create bill IDs
# ------------------------------------------------------------

bill_ids = [
    f"B{i:06d}"
    for i in range(1, N_BILLS + 1)
]

# ------------------------------------------------------------
# Create billing base table
# ------------------------------------------------------------

fact_billing = fact_encounters[
    [
        "encounter_id",
        "patient_id",
        "hospital_id",
        "department_id"
    ]
].copy()

fact_billing.insert(
    0,
    "bill_id",
    bill_ids
)

# ------------------------------------------------------------
# Assign insurance from patient dimension
# ------------------------------------------------------------

patient_insurance = pd.read_csv(
    "../data/raw/dim_patient.csv",
    usecols=["patient_id", "insurance_id"]
)

fact_billing = fact_billing.merge(
    patient_insurance,
    on="patient_id",
    how="left"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Billing base table created!")

print("\nRows:", len(fact_billing))
print("Columns:", len(fact_billing.columns))

print(
    "\nUnique bill IDs:",
    fact_billing["bill_id"].nunique()
)

print(
    "Duplicate bill IDs:",
    fact_billing["bill_id"].duplicated().sum()
)

print(
    "Missing encounter IDs:",
    fact_billing["encounter_id"].isna().sum()
)

print(
    "Missing insurance IDs:",
    fact_billing["insurance_id"].isna().sum()
)

print("\nCurrent columns:")
print(fact_billing.columns.tolist())

Billing base table created!

Rows: 300000
Columns: 6

Unique bill IDs: 300000
Duplicate bill IDs: 0
Missing encounter IDs: 0
Missing insurance IDs: 42400

Current columns:
['bill_id', 'encounter_id', 'patient_id', 'hospital_id', 'department_id', 'insurance_id']


In [17]:
# ============================================================
# CELL 17 — GENERATE BILLED AMOUNTS
# ============================================================

# Bring encounter type and diagnosis into billing table
billing_attributes = fact_encounters[
    [
        "encounter_id",
        "encounter_type",
        "diagnosis_id"
    ]
].copy()

fact_billing = fact_billing.merge(
    billing_attributes,
    on="encounter_id",
    how="left"
)

# ------------------------------------------------------------
# Base billing amount by encounter type
# ------------------------------------------------------------

base_amount = fact_billing["encounter_type"].map({
    "Outpatient": 1800,
    "Inpatient": 18000,
    "Emergency": 7500
})

# ------------------------------------------------------------
# Diagnosis complexity / service intensity
# ------------------------------------------------------------

diagnosis_multiplier = fact_billing["diagnosis_id"].map({
    "DG001": 1.00,  # Hypertension
    "DG002": 1.05,  # Diabetes
    "DG003": 0.90,  # Respiratory infection
    "DG004": 0.85,  # Back pain
    "DG005": 0.90,  # GERD
    "DG006": 1.40,  # Ischemic heart disease
    "DG007": 0.95,  # Migraine
    "DG008": 1.25,  # Fracture
    "DG009": 1.60,  # Breast cancer
    "DG010": 1.70,  # Lung cancer
    "DG011": 0.85,  # Dermatitis
    "DG012": 0.90   # Urinary infection
})

# ------------------------------------------------------------
# Random operational variation
# ------------------------------------------------------------

random_variation = np.random.lognormal(
    mean=0,
    sigma=0.25,
    size=len(fact_billing)
)

# ------------------------------------------------------------
# Final billed amount
# ------------------------------------------------------------

billed_amount = (
    base_amount
    * diagnosis_multiplier
    * random_variation
)

# Round to realistic billing values
fact_billing["billed_amount"] = np.round(
    billed_amount,
    2
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Billed amount generation completed!")

print(
    "\nMissing billed amounts:",
    fact_billing["billed_amount"].isna().sum()
)

print(
    "Minimum billed amount:",
    round(fact_billing["billed_amount"].min(), 2)
)

print(
    "Maximum billed amount:",
    round(fact_billing["billed_amount"].max(), 2)
)

print(
    "Average billed amount:",
    round(fact_billing["billed_amount"].mean(), 2)
)

print(
    "Median billed amount:",
    round(fact_billing["billed_amount"].median(), 2)
)

Billed amount generation completed!

Missing billed amounts: 0
Minimum billed amount: 502.56
Maximum billed amount: 81015.33
Average billed amount: 5937.78
Median billed amount: 2226.53


In [18]:
# ============================================================
# CELL 18 — GENERATE APPROVED AMOUNT
# ============================================================

# Load insurance coverage information
insurance_coverage = dim_insurance[
    [
        "insurance_id",
        "coverage_percentage"
    ]
].copy()

# ------------------------------------------------------------
# Map coverage percentage to each bill
# ------------------------------------------------------------

fact_billing = fact_billing.merge(
    insurance_coverage,
    on="insurance_id",
    how="left"
)

# ------------------------------------------------------------
# Generate approved amount
# ------------------------------------------------------------

# Insured patients:
# approved amount is based on insurance coverage,
# with small claim-level variation.

claim_variation = np.random.uniform(
    0.92,
    1.00,
    size=len(fact_billing)
)

insured_mask = (
    fact_billing["coverage_percentage"].notna()
)

fact_billing["approved_amount"] = np.where(
    insured_mask,
    fact_billing["billed_amount"]
    * fact_billing["coverage_percentage"]
    * claim_variation,
    fact_billing["billed_amount"]
)

# Approved amount cannot exceed billed amount
fact_billing["approved_amount"] = np.minimum(
    fact_billing["approved_amount"],
    fact_billing["billed_amount"]
)

fact_billing["approved_amount"] = np.round(
    fact_billing["approved_amount"],
    2
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Approved amount generation completed!")

print(
    "\nMissing approved amounts:",
    fact_billing["approved_amount"].isna().sum()
)

print(
    "Approved amount > billed amount:",
    (
        fact_billing["approved_amount"]
        > fact_billing["billed_amount"]
    ).sum()
)

print(
    "Average approved amount:",
    round(
        fact_billing["approved_amount"].mean(),
        2
    )
)

print(
    "Median approved amount:",
    round(
        fact_billing["approved_amount"].median(),
        2
    )
)

print(
    "\nBills with missing insurance:",
    fact_billing["coverage_percentage"].isna().sum()
)

Approved amount generation completed!

Missing approved amounts: 0
Approved amount > billed amount: 0
Average approved amount: 4832.99
Median approved amount: 1833.74

Bills with missing insurance: 42400


In [19]:
# ============================================================
# CELL 19 — GENERATE PAYMENT ALLOCATION
# ============================================================

# ------------------------------------------------------------
# Generate collection behavior
# ------------------------------------------------------------

n = len(fact_billing)

payment_random = np.random.random(n)

# Collection rate against approved amount
collection_rate = np.select(
    [
        payment_random < 0.72,
        payment_random < 0.92
    ],
    [
        np.random.uniform(0.95, 1.00, n),
        np.random.uniform(0.50, 0.95, n)
    ],
    default=np.random.uniform(0.00, 0.50, n)
)

# ------------------------------------------------------------
# Total amount collected
# ------------------------------------------------------------

total_collected = (
    fact_billing["approved_amount"]
    * collection_rate
)

total_collected = np.minimum(
    total_collected,
    fact_billing["approved_amount"]
)

# ------------------------------------------------------------
# Split collected amount between insurer and patient
# ------------------------------------------------------------

insured_mask = (
    fact_billing["coverage_percentage"].notna()
)

# Insurance share of collected amount
insurance_share = np.where(
    insured_mask,
    fact_billing["coverage_percentage"],
    0
)

fact_billing["insurance_paid"] = (
    total_collected * insurance_share
)

fact_billing["patient_paid"] = (
    total_collected
    - fact_billing["insurance_paid"]
)

# Outstanding balance
fact_billing["outstanding_amount"] = (
    fact_billing["approved_amount"]
    - fact_billing["insurance_paid"]
    - fact_billing["patient_paid"]
)

# Round financial values
financial_columns = [
    "insurance_paid",
    "patient_paid",
    "outstanding_amount"
]

fact_billing[financial_columns] = (
    fact_billing[financial_columns]
    .round(2)
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Payment allocation completed!")

print(
    "\nMissing insurance payments:",
    fact_billing["insurance_paid"].isna().sum()
)

print(
    "Missing patient payments:",
    fact_billing["patient_paid"].isna().sum()
)

print(
    "Missing outstanding amounts:",
    fact_billing["outstanding_amount"].isna().sum()
)

# Financial consistency
balance_difference = (
    fact_billing["approved_amount"]
    - fact_billing["insurance_paid"]
    - fact_billing["patient_paid"]
    - fact_billing["outstanding_amount"]
)

print(
    "\nFinancial consistency failures:",
    (abs(balance_difference) > 0.01).sum()
)

print(
    "Negative outstanding balances:",
    (fact_billing["outstanding_amount"] < 0).sum()
)

print(
    "\nTotal billed:",
    round(fact_billing["billed_amount"].sum(), 2)
)

print(
    "Total approved:",
    round(fact_billing["approved_amount"].sum(), 2)
)

print(
    "Total insurance paid:",
    round(fact_billing["insurance_paid"].sum(), 2)
)

print(
    "Total patient paid:",
    round(fact_billing["patient_paid"].sum(), 2)
)

print(
    "Total outstanding:",
    round(fact_billing["outstanding_amount"].sum(), 2)
)

Payment allocation completed!

Missing insurance payments: 0
Missing patient payments: 0
Missing outstanding amounts: 0

Financial consistency failures: 33932
Negative outstanding balances: 0

Total billed: 1781335181.01
Total approved: 1449897545.66
Total insurance paid: 863786322.28
Total patient paid: 393046891.82
Total outstanding: 193064332.42


In [20]:
# ============================================================
# CELL 19A — DIAGNOSE FINANCIAL CONSISTENCY
# ============================================================

balance_difference = (
    fact_billing["approved_amount"]
    - fact_billing["insurance_paid"]
    - fact_billing["patient_paid"]
    - fact_billing["outstanding_amount"]
)

print("Financial discrepancy analysis")
print("--------------------------------")

print(
    "Max absolute discrepancy:",
    round(balance_difference.abs().max(), 4)
)

print(
    "Mean absolute discrepancy:",
    round(balance_difference.abs().mean(), 4)
)

print(
    "Median absolute discrepancy:",
    round(balance_difference.abs().median(), 4)
)

print(
    "Discrepancies > ₹0.01:",
    (balance_difference.abs() > 0.01).sum()
)

print(
    "Discrepancies > ₹1:",
    (balance_difference.abs() > 1).sum()
)

print(
    "Discrepancies > ₹10:",
    (balance_difference.abs() > 10).sum()
)

print(
    "Discrepancies > ₹100:",
    (balance_difference.abs() > 100).sum()
)

print("\nLargest discrepancies:")

diagnostic = fact_billing[
    [
        "bill_id",
        "approved_amount",
        "insurance_paid",
        "patient_paid",
        "outstanding_amount"
    ]
].copy()

diagnostic["difference"] = balance_difference

print(
    diagnostic
    .loc[diagnostic["difference"].abs().nlargest(10).index]
    .sort_values("difference", key=lambda x: x.abs(), ascending=False)
    .to_string(index=False)
)

Financial discrepancy analysis
--------------------------------
Max absolute discrepancy: 0.01
Mean absolute discrepancy: 0.0024
Median absolute discrepancy: 0.0
Discrepancies > ₹0.01: 33932
Discrepancies > ₹1: 0
Discrepancies > ₹10: 0
Discrepancies > ₹100: 0

Largest discrepancies:
bill_id  approved_amount  insurance_paid  patient_paid  outstanding_amount  difference
B041065         36131.09         9444.33       1049.37            25637.40       -0.01
B195251         39924.09        24470.40       1287.92            14165.78       -0.01
B083655         37053.24        33905.98       1784.53             1362.74       -0.01
B153620         40779.48        29178.01       9726.00             1875.46        0.01
B190087         36306.87        25831.01       4558.41             5917.44        0.01
B147238         41261.98        36822.53       4091.39              348.05        0.01
B296017         37077.84        35176.54       1851.40               49.91       -0.01
B043333         4328

In [21]:
# ============================================================
# CELL 21B — FINAL PAYMENT ALLOCATION CORRECTION
# ============================================================

approved = fact_billing["approved_amount"].to_numpy()
coverage = fact_billing["coverage_percentage"].fillna(0).to_numpy()

# ------------------------------------------------------------
# Recalculate collected amount
# ------------------------------------------------------------

payment_random = np.random.random(len(fact_billing))

collection_rate = np.select(
    [
        payment_random < 0.70,
        payment_random < 0.92
    ],
    [
        1.00,
        np.random.uniform(0.50, 0.90, len(fact_billing))
    ],
    default=np.random.uniform(0.00, 0.20, len(fact_billing))
)

total_collected = np.round(
    approved * collection_rate,
    2
)

# Never allow collection above approved amount
total_collected = np.minimum(
    total_collected,
    approved
)

# ------------------------------------------------------------
# Insurance payment
# ------------------------------------------------------------

insurance_paid = np.round(
    total_collected * coverage,
    2
)

# Insurance payment cannot exceed total collected
insurance_paid = np.minimum(
    insurance_paid,
    total_collected
)

# ------------------------------------------------------------
# Patient payment
# ------------------------------------------------------------

patient_paid = np.round(
    total_collected - insurance_paid,
    2
)

# Prevent patient payment from exceeding remaining approved amount
patient_paid = np.minimum(
    patient_paid,
    approved - insurance_paid
)

# ------------------------------------------------------------
# Outstanding amount
# ------------------------------------------------------------

outstanding = np.round(
    approved
    - insurance_paid
    - patient_paid,
    2
)

# Final safety check
outstanding = np.maximum(
    outstanding,
    0
)

# ------------------------------------------------------------
# Store final values
# ------------------------------------------------------------

fact_billing["insurance_paid"] = insurance_paid
fact_billing["patient_paid"] = patient_paid
fact_billing["outstanding_amount"] = outstanding

# ------------------------------------------------------------
# Generate final payment status
# ------------------------------------------------------------

fact_billing["payment_status"] = np.select(
    [
        fact_billing["outstanding_amount"] <= 0.01,
        fact_billing["outstanding_amount"] > 0.01
    ],
    [
        "Paid",
        "Partial"
    ],
    default="Outstanding"
)

# Force truly unpaid bills to Outstanding
fact_billing.loc[
    (
        fact_billing["insurance_paid"]
        + fact_billing["patient_paid"]
    ) <= 0.01,
    "payment_status"
] = "Outstanding"

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

balance_difference = (
    fact_billing["approved_amount"]
    - fact_billing["insurance_paid"]
    - fact_billing["patient_paid"]
    - fact_billing["outstanding_amount"]
)

print("Final payment allocation correction completed!")

print("\nPayment status distribution:")
print(
    fact_billing["payment_status"]
    .value_counts()
)

print("\nPayment status percentages:")
print(
    (
        fact_billing["payment_status"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

print(
    "\nFinancial consistency failures:",
    (balance_difference.abs() > 0.01).sum()
)

print(
    "Negative outstanding balances:",
    (fact_billing["outstanding_amount"] < 0).sum()
)

print(
    "Maximum financial discrepancy:",
    round(balance_difference.abs().max(), 4)
)

print(
    "\nTotal approved:",
    round(fact_billing["approved_amount"].sum(), 2)
)

print(
    "Total insurance paid:",
    round(fact_billing["insurance_paid"].sum(), 2)
)

print(
    "Total patient paid:",
    round(fact_billing["patient_paid"].sum(), 2)
)

print(
    "Total outstanding:",
    round(fact_billing["outstanding_amount"].sum(), 2)
)

Final payment allocation correction completed!

Payment status distribution:
payment_status
Paid           209970
Partial         90029
Outstanding         1
Name: count, dtype: int64

Payment status percentages:
payment_status
Paid           69.99
Partial        30.01
Outstanding     0.00
Name: proportion, dtype: float64

Financial consistency failures: 0
Negative outstanding balances: 0
Maximum financial discrepancy: 0.0

Total approved: 1449897545.66
Total insurance paid: 859650895.06
Total patient paid: 389812784.85
Total outstanding: 200433865.75


In [22]:
# ============================================================
# CELL 20 — GENERATE CLAIM STATUS
# ============================================================

n = len(fact_billing)

# Random values for claim processing behavior
claim_random = np.random.random(n)

insured_mask = (
    fact_billing["insurance_id"].notna()
)

# Default: uninsured bills are treated as approved/self-pay
fact_billing["claim_status"] = "Approved"

# Apply claim outcomes only to insured bills
fact_billing.loc[
    insured_mask & (claim_random < 0.05),
    "claim_status"
] = "Rejected"

fact_billing.loc[
    insured_mask
    & (claim_random >= 0.05)
    & (claim_random < 0.15),
    "claim_status"
] = "Pending"

fact_billing.loc[
    insured_mask
    & (claim_random >= 0.15)
    & (claim_random < 0.35),
    "claim_status"
] = "Partial"

fact_billing.loc[
    insured_mask & (claim_random >= 0.35),
    "claim_status"
] = "Approved"

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Claim status generation completed!")

print("\nClaim status distribution:")
print(
    fact_billing["claim_status"]
    .value_counts()
)

print("\nClaim status percentages:")
print(
    (
        fact_billing["claim_status"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

print(
    "\nUninsured bills:",
    (~insured_mask).sum()
)

print(
    "Uninsured bills marked Approved:",
    (
        (~insured_mask)
        & (fact_billing["claim_status"] == "Approved")
    ).sum()
)

print(
    "Missing claim status:",
    fact_billing["claim_status"].isna().sum()
)

Claim status generation completed!

Claim status distribution:
claim_status
Approved    210071
Partial      51352
Pending      25702
Rejected     12875
Name: count, dtype: int64

Claim status percentages:
claim_status
Approved    70.02
Partial     17.12
Pending      8.57
Rejected     4.29
Name: proportion, dtype: float64

Uninsured bills: 42400
Uninsured bills marked Approved: 42400
Missing claim status: 0


In [23]:
# ============================================================
# CELL 21 — GENERATE PAYMENT STATUS
# ============================================================

fact_billing["payment_status"] = np.select(
    [
        fact_billing["outstanding_amount"] <= 0.01,
        (
            fact_billing["outstanding_amount"] > 0.01
        )
        & (
            fact_billing["insurance_paid"]
            + fact_billing["patient_paid"]
            > 0
        )
    ],
    [
        "Paid",
        "Partial"
    ],
    default="Outstanding"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Payment status generation completed!")

print("\nPayment status distribution:")
print(
    fact_billing["payment_status"]
    .value_counts()
)

print("\nPayment status percentages:")
print(
    (
        fact_billing["payment_status"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

print(
    "\nMissing payment status:",
    fact_billing["payment_status"].isna().sum()
)

# ------------------------------------------------------------
# Financial consistency by payment status
# ------------------------------------------------------------

print("\nAverage outstanding amount by payment status:")

print(
    fact_billing
    .groupby("payment_status")["outstanding_amount"]
    .mean()
    .round(2)
)

Payment status generation completed!

Payment status distribution:
payment_status
Paid           209970
Partial         90029
Outstanding         1
Name: count, dtype: int64

Payment status percentages:
payment_status
Paid           69.99
Partial        30.01
Outstanding     0.00
Name: proportion, dtype: float64

Missing payment status: 0

Average outstanding amount by payment status:
payment_status
Outstanding     939.91
Paid              0.00
Partial        2226.32
Name: outstanding_amount, dtype: float64


In [24]:
# ============================================================
# CELL 22 — GENERATE BILL DATE ID
# ============================================================

# Load date dimension
dim_date["full_date"] = pd.to_datetime(
    dim_date["full_date"]
)

# Load encounter discharge dates
encounter_dates = fact_encounters[
    [
        "encounter_id",
        "discharge_datetime"
    ]
].copy()

encounter_dates["discharge_datetime"] = pd.to_datetime(
    encounter_dates["discharge_datetime"]
)

# ------------------------------------------------------------
# Merge discharge date into billing table
# ------------------------------------------------------------

fact_billing = fact_billing.merge(
    encounter_dates,
    on="encounter_id",
    how="left"
)

# ------------------------------------------------------------
# Derive bill date
# ------------------------------------------------------------

fact_billing["bill_date"] = (
    fact_billing["discharge_datetime"]
    .dt.normalize()
)

# ------------------------------------------------------------
# Map bill date to date_id
# ------------------------------------------------------------

date_mapping = dim_date[
    [
        "date_id",
        "full_date"
    ]
].copy()

fact_billing = fact_billing.merge(
    date_mapping,
    left_on="bill_date",
    right_on="full_date",
    how="left"
)

fact_billing = fact_billing.rename(
    columns={
        "date_id": "bill_date_id"
    }
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Bill date generation completed!")

print(
    "\nMissing bill dates:",
    fact_billing["bill_date"].isna().sum()
)

print(
    "Missing bill date IDs:",
    fact_billing["bill_date_id"].isna().sum()
)

print(
    "Unique bill dates:",
    fact_billing["bill_date"].nunique()
)

print(
    "Unique bill date IDs:",
    fact_billing["bill_date_id"].nunique()
)

print(
    "\nBill date range:",
    fact_billing["bill_date"].min(),
    "to",
    fact_billing["bill_date"].max()
)

print(
    "\nRows:",
    len(fact_billing)
)

Bill date generation completed!

Missing bill dates: 0
Missing bill date IDs: 9
Unique bill dates: 731
Unique bill date IDs: 730

Bill date range: 2024-01-02 00:00:00 to 2026-01-01 00:00:00

Rows: 300000


In [26]:
# ============================================================
# CELL 22 — GENERATE BILL DATE ID
# ============================================================

# ------------------------------------------------------------
# Load date dimension
# ------------------------------------------------------------

dim_date["full_date"] = pd.to_datetime(
    dim_date["full_date"]
)

# ------------------------------------------------------------
# Load encounter discharge dates
# ------------------------------------------------------------

encounter_dates = fact_encounters[
    [
        "encounter_id",
        "discharge_datetime"
    ]
].copy()

encounter_dates["discharge_datetime"] = pd.to_datetime(
    encounter_dates["discharge_datetime"]
)

# ------------------------------------------------------------
# Remove old temporary date columns if they already exist
# ------------------------------------------------------------

for col in [
    "discharge_datetime",
    "bill_date",
    "full_date",
    "bill_date_id"
]:
    if col in fact_billing.columns:
        fact_billing = fact_billing.drop(
            columns=col
        )

# ------------------------------------------------------------
# Merge discharge date
# ------------------------------------------------------------

fact_billing = fact_billing.merge(
    encounter_dates,
    on="encounter_id",
    how="left"
)

# ------------------------------------------------------------
# Derive bill date
# ------------------------------------------------------------

fact_billing["bill_date"] = (
    fact_billing["discharge_datetime"]
    .dt.normalize()
)

# ------------------------------------------------------------
# Map bill date to date_id
# ------------------------------------------------------------

date_mapping = dim_date[
    [
        "date_id",
        "full_date"
    ]
].copy()

fact_billing = fact_billing.merge(
    date_mapping,
    left_on="bill_date",
    right_on="full_date",
    how="left"
)

fact_billing = fact_billing.rename(
    columns={
        "date_id": "bill_date_id"
    }
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Bill date generation completed!")

print(
    "\nMissing bill dates:",
    fact_billing["bill_date"].isna().sum()
)

print(
    "Missing bill date IDs:",
    fact_billing["bill_date_id"].isna().sum()
)

print(
    "Unique bill dates:",
    fact_billing["bill_date"].nunique()
)

print(
    "Unique bill date IDs:",
    fact_billing["bill_date_id"].nunique()
)

print(
    "\nBill date range:",
    fact_billing["bill_date"].min(),
    "to",
    fact_billing["bill_date"].max()
)

print(
    "\nRows:",
    len(fact_billing)
)

Bill date generation completed!

Missing bill dates: 0
Missing bill date IDs: 9
Unique bill dates: 731
Unique bill date IDs: 730

Bill date range: 2024-01-02 00:00:00 to 2026-01-01 00:00:00

Rows: 300000


In [27]:
# ============================================================
# CELL 22B — HANDLE BILLING DATES OUTSIDE PROJECT PERIOD
# ============================================================

# Bills discharged on 2026-01-01 are assigned to the
# final reporting date of the project: 2025-12-31.

outside_period_mask = (
    fact_billing["bill_date"]
    > dim_date["full_date"].max()
)

print(
    "Bills outside project date range:",
    outside_period_mask.sum()
)

# Assign them to final project date
fact_billing.loc[
    outside_period_mask,
    "bill_date"
] = dim_date["full_date"].max()

# ------------------------------------------------------------
# Re-map bill date to date_id
# ------------------------------------------------------------

date_mapping = dim_date[
    [
        "date_id",
        "full_date"
    ]
].copy()

fact_billing = fact_billing.drop(
    columns=["bill_date_id", "full_date"]
)

fact_billing = fact_billing.merge(
    date_mapping,
    left_on="bill_date",
    right_on="full_date",
    how="left"
)

fact_billing = fact_billing.rename(
    columns={
        "date_id": "bill_date_id"
    }
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print(
    "\nMissing bill date IDs:",
    fact_billing["bill_date_id"].isna().sum()
)

print(
    "Bill date range:",
    fact_billing["bill_date"].min(),
    "to",
    fact_billing["bill_date"].max()
)

print(
    "Unique bill date IDs:",
    fact_billing["bill_date_id"].nunique()
)

print(
    "Rows:",
    len(fact_billing)
)

Bills outside project date range: 9

Missing bill date IDs: 0
Bill date range: 2024-01-02 00:00:00 to 2025-12-31 00:00:00
Unique bill date IDs: 730
Rows: 300000


In [28]:
# ============================================================
# CELL 23 — GENERATE CLAIM STATUS
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(42)

# ------------------------------------------------------------
# Identify insured and uninsured bills
# ------------------------------------------------------------

insured_mask = fact_billing["insurance_id"].notna()
uninsured_mask = fact_billing["insurance_id"].isna()

# ------------------------------------------------------------
# Initialize claim status
# ------------------------------------------------------------

fact_billing["claim_status"] = "Approved"

# ------------------------------------------------------------
# Uninsured bills
# ------------------------------------------------------------
# These are treated as self-pay.
# No insurance claim is required.
# ------------------------------------------------------------

fact_billing.loc[
    uninsured_mask,
    "claim_status"
] = "Approved"

# ------------------------------------------------------------
# Generate claim outcomes for insured bills
# ------------------------------------------------------------

insured_indices = fact_billing.index[insured_mask]

claim_random = np.random.random(
    len(insured_indices)
)

# 5% Rejected
fact_billing.loc[
    insured_indices[claim_random < 0.05],
    "claim_status"
] = "Rejected"

# 10% Pending
fact_billing.loc[
    insured_indices[
        (claim_random >= 0.05) &
        (claim_random < 0.15)
    ],
    "claim_status"
] = "Pending"

# 20% Partial
fact_billing.loc[
    insured_indices[
        (claim_random >= 0.15) &
        (claim_random < 0.35)
    ],
    "claim_status"
] = "Partial"

# 65% Approved
fact_billing.loc[
    insured_indices[
        claim_random >= 0.35
    ],
    "claim_status"
] = "Approved"

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("Claim status generation completed!")

print("\nClaim Status Distribution:")
print(
    fact_billing["claim_status"]
    .value_counts()
    .sort_index()
)

print("\nClaim Status Percentage:")
print(
    (
        fact_billing["claim_status"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

print(
    "\nMissing claim status:",
    fact_billing["claim_status"].isna().sum()
)

print(
    "Insured bills:",
    insured_mask.sum()
)

print(
    "Uninsured bills:",
    uninsured_mask.sum()
)

print(
    "Uninsured bills marked Approved:",
    (
        fact_billing.loc[
            uninsured_mask,
            "claim_status"
        ] == "Approved"
    ).sum()
)

Claim status generation completed!

Claim Status Distribution:
claim_status
Approved    209880
Partial      51583
Pending      25843
Rejected     12694
Name: count, dtype: int64

Claim Status Percentage:
claim_status
Approved    69.96
Partial     17.19
Pending      8.61
Rejected     4.23
Name: proportion, dtype: float64

Missing claim status: 0
Insured bills: 257600
Uninsured bills: 42400
Uninsured bills marked Approved: 42400


In [30]:
# ============================================================
# CELL 24 — GENERATE PAYMENT ALLOCATION & PAYMENT STATUS
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(2026)

# ------------------------------------------------------------
# Initialize payment columns
# ------------------------------------------------------------

fact_billing["insurance_paid"] = 0.0
fact_billing["patient_paid"] = 0.0
fact_billing["outstanding_amount"] = 0.0

# ------------------------------------------------------------
# Process each bill
# ------------------------------------------------------------

for idx, row in fact_billing.iterrows():

    approved = float(row["approved_amount"])
    claim_status = row["claim_status"]
    insurance_id = row["insurance_id"]

    # Reset for every row
    insurance_paid = 0.0
    patient_paid = 0.0

    # ========================================================
    # 1. UNINSURED / SELF-PAY
    # ========================================================

    if pd.isna(insurance_id):

        r = np.random.random()

        # 65% fully paid
        if r < 0.65:

            patient_paid = approved

        # 25% partially paid
        elif r < 0.90:

            patient_paid = (
                approved
                * np.random.uniform(0.30, 0.85)
            )

        # 10% completely outstanding
        else:

            patient_paid = 0.0

    # ========================================================
    # 2. REJECTED INSURANCE CLAIM
    # ========================================================

    elif claim_status == "Rejected":

        # Insurance pays nothing
        insurance_paid = 0.0

        r = np.random.random()

        # Some patients pay part of the bill
        if r < 0.35:

            patient_paid = (
                approved
                * np.random.uniform(0.20, 0.70)
            )

        else:

            patient_paid = 0.0

    # ========================================================
    # 3. PENDING INSURANCE CLAIM
    # ========================================================

    elif claim_status == "Pending":

        # Insurance has not paid yet
        insurance_paid = 0.0

        r = np.random.random()

        # Patient may pay some amount
        if r < 0.45:

            patient_paid = (
                approved
                * np.random.uniform(0.10, 0.40)
            )

        else:

            patient_paid = 0.0

    # ========================================================
    # 4. PARTIAL INSURANCE CLAIM
    # ========================================================

    elif claim_status == "Partial":

        # Insurance pays part of approved amount
        insurance_paid = (
            approved
            * np.random.uniform(0.30, 0.70)
        )

        remaining_amount = (
            approved - insurance_paid
        )

        r = np.random.random()

        # Patient pays part or all of remaining amount
        if r < 0.70:

            patient_paid = (
                remaining_amount
                * np.random.uniform(0.30, 1.00)
            )

        else:

            patient_paid = 0.0

    # ========================================================
    # 5. APPROVED INSURANCE CLAIM
    # ========================================================

    elif claim_status == "Approved":

        # Insurance pays majority of approved amount
        insurance_paid = (
            approved
            * np.random.uniform(0.55, 0.95)
        )

        remaining_amount = (
            approved - insurance_paid
        )

        r = np.random.random()

        # Patient usually pays remaining portion
        if r < 0.85:

            patient_paid = (
                remaining_amount
                * np.random.uniform(0.70, 1.00)
            )

        else:

            patient_paid = 0.0

    # --------------------------------------------------------
    # Safety controls
    # --------------------------------------------------------

    insurance_paid = min(
        max(insurance_paid, 0.0),
        approved
    )

    patient_paid = min(
        max(patient_paid, 0.0),
        approved - insurance_paid
    )

    # --------------------------------------------------------
    # Round payments
    # --------------------------------------------------------

    insurance_paid = round(
        insurance_paid,
        2
    )

    patient_paid = round(
        patient_paid,
        2
    )

    # --------------------------------------------------------
    # Calculate outstanding after rounding
    # --------------------------------------------------------

    outstanding_amount = round(
        approved
        - insurance_paid
        - patient_paid,
        2
    )

    # Final protection against floating-point issues
    if outstanding_amount < 0:

        outstanding_amount = 0.0

        patient_paid = round(
            approved - insurance_paid,
            2
        )

    # --------------------------------------------------------
    # Store values
    # --------------------------------------------------------

    fact_billing.at[
        idx,
        "insurance_paid"
    ] = insurance_paid

    fact_billing.at[
        idx,
        "patient_paid"
    ] = patient_paid

    fact_billing.at[
        idx,
        "outstanding_amount"
    ] = outstanding_amount


# ============================================================
# GENERATE PAYMENT STATUS
# ============================================================

fact_billing["payment_status"] = np.select(
    [
        fact_billing["outstanding_amount"] <= 0.01,

        (
            fact_billing["outstanding_amount"] > 0.01
        )
        &
        (
            (
                fact_billing["insurance_paid"]
                +
                fact_billing["patient_paid"]
            ) > 0
        )
    ],
    [
        "Paid",
        "Partial"
    ],
    default="Outstanding"
)

# ============================================================
# FINANCIAL VALIDATION
# ============================================================

fact_billing["financial_difference"] = (
    fact_billing["approved_amount"]
    -
    (
        fact_billing["insurance_paid"]
        +
        fact_billing["patient_paid"]
        +
        fact_billing["outstanding_amount"]
    )
)

print("Payment allocation completed!")

print("\nPayment Status Distribution:")
print(
    fact_billing["payment_status"]
    .value_counts()
    .sort_index()
)

print("\nPayment Status Percentage:")
print(
    (
        fact_billing["payment_status"]
        .value_counts(normalize=True)
        * 100
    ).round(2)
)

print(
    "\nFinancial consistency failures:",
    (
        fact_billing["financial_difference"].abs()
        > 0.01
    ).sum()
)

print(
    "Maximum financial difference:",
    round(
        fact_billing["financial_difference"]
        .abs()
        .max(),
        4
    )
)

print(
    "Negative insurance payments:",
    (
        fact_billing["insurance_paid"] < 0
    ).sum()
)

print(
    "Negative patient payments:",
    (
        fact_billing["patient_paid"] < 0
    ).sum()
)

print(
    "Negative outstanding amounts:",
    (
        fact_billing["outstanding_amount"] < 0
    ).sum()
)

print(
    "\nTotal approved:",
    round(
        fact_billing["approved_amount"].sum(),
        2
    )
)

print(
    "Total insurance paid:",
    round(
        fact_billing["insurance_paid"].sum(),
        2
    )
)

print(
    "Total patient paid:",
    round(
        fact_billing["patient_paid"].sum(),
        2
    )
)

print(
    "Total outstanding:",
    round(
        fact_billing["outstanding_amount"].sum(),
        2
    )
)

Payment allocation completed!

Payment Status Distribution:
payment_status
Outstanding     26826
Paid            27414
Partial        245760
Name: count, dtype: int64

Payment Status Percentage:
payment_status
Partial        81.92
Paid            9.14
Outstanding     8.94
Name: proportion, dtype: float64

Financial consistency failures: 0
Maximum financial difference: 0.0
Negative insurance payments: 0
Negative patient payments: 0
Negative outstanding amounts: 0

Total approved: 1449897545.66
Total insurance paid: 703368798.26
Total patient paid: 416462470.05
Total outstanding: 330066277.35


In [31]:
# FINAL BILLING VALIDATION

print("FACT BILLING VALIDATION")
print("=" * 60)

# 1. Row and ID checks
print("\n1. STRUCTURE")
print("Rows:", len(fact_billing))
print("Unique bill IDs:", fact_billing["bill_id"].nunique())
print("Unique encounter IDs:", fact_billing["encounter_id"].nunique())

# 2. Required missing values
required_cols = [
    "bill_id", "encounter_id", "patient_id",
    "hospital_id", "department_id",
    "bill_date_id", "billed_amount",
    "approved_amount", "insurance_paid",
    "patient_paid", "outstanding_amount",
    "claim_status", "payment_status"
]

print("\n2. REQUIRED MISSING VALUES")
print(fact_billing[required_cols].isna().sum())

# 3. Financial consistency
financial_diff = (
    fact_billing["approved_amount"]
    - fact_billing["insurance_paid"]
    - fact_billing["patient_paid"]
    - fact_billing["outstanding_amount"]
)

print("\n3. FINANCIAL CONSISTENCY")
print("Failures:", (financial_diff.abs() > 0.01).sum())
print("Maximum difference:", financial_diff.abs().max())

# 4. Financial validity
print("\n4. FINANCIAL VALIDITY")
print("Negative billed:", (fact_billing["billed_amount"] < 0).sum())
print("Negative approved:", (fact_billing["approved_amount"] < 0).sum())
print("Negative insurance paid:", (fact_billing["insurance_paid"] < 0).sum())
print("Negative patient paid:", (fact_billing["patient_paid"] < 0).sum())
print("Negative outstanding:", (fact_billing["outstanding_amount"] < 0).sum())

print(
    "Approved > Billed:",
    (fact_billing["approved_amount"] > fact_billing["billed_amount"] + 0.01).sum()
)

# 5. Claim status
print("\n5. CLAIM STATUS")
print(fact_billing["claim_status"].value_counts())
print(
    (fact_billing["claim_status"].value_counts(normalize=True) * 100)
    .round(2)
)

# 6. Payment status
print("\n6. PAYMENT STATUS")
print(fact_billing["payment_status"].value_counts())
print(
    (fact_billing["payment_status"].value_counts(normalize=True) * 100)
    .round(2)
)

# 7. Payment status consistency
payment_status_check = (
    ((fact_billing["outstanding_amount"] <= 0.01) &
     (fact_billing["payment_status"] != "Paid")) |
    ((fact_billing["outstanding_amount"] > 0.01) &
     (fact_billing["payment_status"] == "Paid"))
)

print("\n7. PAYMENT STATUS CONSISTENCY")
print("Failures:", payment_status_check.sum())

# 8. Insurance coverage
print("\n8. INSURANCE COVERAGE")
print("Bills with insurance:", fact_billing["insurance_id"].notna().sum())
print("Bills without insurance:", fact_billing["insurance_id"].isna().sum())

print("\n" + "=" * 60)
print("FINAL BILLING VALIDATION COMPLETE")

FACT BILLING VALIDATION

1. STRUCTURE
Rows: 300000
Unique bill IDs: 300000
Unique encounter IDs: 300000

2. REQUIRED MISSING VALUES
bill_id               0
encounter_id          0
patient_id            0
hospital_id           0
department_id         0
bill_date_id          0
billed_amount         0
approved_amount       0
insurance_paid        0
patient_paid          0
outstanding_amount    0
claim_status          0
payment_status        0
dtype: int64

3. FINANCIAL CONSISTENCY
Failures: 0
Maximum difference: 7.275957614183426e-12

4. FINANCIAL VALIDITY
Negative billed: 0
Negative approved: 0
Negative insurance paid: 0
Negative patient paid: 0
Negative outstanding: 0
Approved > Billed: 0

5. CLAIM STATUS
claim_status
Approved    209880
Partial      51583
Pending      25843
Rejected     12694
Name: count, dtype: int64
claim_status
Approved    69.96
Partial     17.19
Pending      8.61
Rejected     4.23
Name: proportion, dtype: float64

6. PAYMENT STATUS
payment_status
Partial        2457

In [32]:
# FINAL TABLE CHECK

tables = {
    "dim_date": dim_date,
    "dim_hospital": dim_hospital,
    "dim_department": dim_department,
    "dim_doctor": dim_doctor,
    "dim_patient": dim_patient,
    "dim_insurance": dim_insurance,
    "dim_diagnosis": dim_diagnosis,
    "fact_appointments": fact_appointments,
    "fact_encounters": fact_encounters,
    "fact_billing": fact_billing,
    "fact_patient_feedback": fact_patient_feedback
}

for name, df in tables.items():
    print(f"{name:25} {df.shape}")

NameError: name 'dim_patient' is not defined

In [33]:
# CHECK CURRENT DATAFRAME VARIABLES

[name for name in globals() if not name.startswith("_")]

['In',
 'Out',
 'get_ipython',
 'exit',
 'quit',
 'open',
 'pd',
 'np',
 'dim_date',
 'dim_doctor',
 'dim_department',
 'dim_diagnosis',
 'fact_appointments',
 'completed_appointments',
 'N_ENCOUNTERS',
 'selected_appointments',
 'fact_encounters',
 'diagnosis_ids',
 'diagnosis_probabilities',
 'arrival_offset_minutes',
 'base_wait',
 'department_capacity',
 'capacity_factor',
 'random_wait',
 'wait_time',
 'high_wait_mask',
 'base_consultation',
 'diagnosis_complexity',
 'random_consultation',
 'consultation_time',
 'post_consultation_duration',
 'random_duration',
 'total_post_consultation',
 'invalid_timeline',
 'admission_probability',
 'random_status',
 'readmission_check',
 'readmission_flags',
 'admitted_encounters',
 'readmissions',
 'admission_count',
 'readmission_rate',
 'required_columns',
 'appointment_mismatch',
 'output_path',
 'encounters_path',
 'fact_encounters_check',
 'dim_hospital',
 'dim_insurance',
 'N_BILLS',
 'bill_ids',
 'fact_billing',
 'patient_insurance',
 

In [34]:
# RELOAD DIM_PATIENT

dim_patient = pd.read_csv(
    r"C:\Users\janak\MediCore_Healthcare_Analytics\data\raw\dim_patient.csv"
)

print("dim_patient shape:", dim_patient.shape)
print("\nColumns:")
print(dim_patient.columns.tolist())

print("\nMissing values:")
print(dim_patient.isna().sum())

dim_patient shape: (100000, 10)

Columns:
['patient_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'city', 'state', 'registration_date', 'insurance_id', 'patient_status']

Missing values:
patient_id               0
first_name               0
last_name                0
gender                   0
date_of_birth            0
city                     0
state                    0
registration_date        0
insurance_id         14068
patient_status           0
dtype: int64


In [35]:
tables = {
    "dim_date": dim_date,
    "dim_hospital": dim_hospital,
    "dim_department": dim_department,
    "dim_doctor": dim_doctor,
    "dim_patient": dim_patient,
    "dim_insurance": dim_insurance,
    "dim_diagnosis": dim_diagnosis,
    "fact_appointments": fact_appointments,
    "fact_encounters": fact_encounters,
    "fact_billing": fact_billing,
    "fact_patient_feedback": fact_patient_feedback
}

for name, df in tables.items():
    print(f"{name:25} {df.shape}")

NameError: name 'fact_patient_feedback' is not defined

In [36]:
import os

raw_path = r"C:\Users\janak\MediCore_Healthcare_Analytics\data\raw"

print(os.listdir(raw_path))

['dim_date.csv', 'dim_department.csv', 'dim_diagnosis.csv', 'dim_doctor.csv', 'dim_hospital.csv', 'dim_insurance.csv', 'dim_patient.csv', 'fact_appointments.csv', 'fact_encounters.csv']


In [37]:
C:\Users\janak\MediCore_Healthcare_Analytics\data\processed\

SyntaxError: unexpected character after line continuation character (3404695303.py, line 1)

In [39]:
import os

processed_path = r"C:\Users\janak\MediCore_Healthcare_Analytics\data\processed"

print(os.listdir(processed_path))

[]


In [40]:
print(fact_encounters.columns.tolist())

['encounter_id', 'appointment_id', 'patient_id', 'doctor_id', 'department_id', 'hospital_id', 'appointment_date_id', 'scheduled_datetime', 'encounter_type', 'diagnosis_id', 'arrival_datetime', 'consultation_start_datetime', 'wait_time_minutes', 'consultation_time_minutes', 'discharge_datetime', 'admission_flag', 'discharge_status', 'readmission_30d_flag']


In [41]:
# GENERATE FACT_PATIENT_FEEDBACK

np.random.seed(2027)

# Use a subset of completed encounters for patients who actually received care
feedback_rate = 0.70
n_feedback = int(len(fact_encounters) * feedback_rate)

feedback_encounters = fact_encounters.sample(
    n=n_feedback,
    random_state=2027
).copy()

feedback_encounters = feedback_encounters.reset_index(drop=True)

# IDs
feedback_encounters["feedback_id"] = [
    f"FB{i:06d}" for i in range(1, n_feedback + 1)
]

# Base experience score
# Longer waits reduce satisfaction
wait_penalty = (
    (feedback_encounters["wait_time_minutes"] - 45) / 30
).clip(lower=0)

# Consultation time gives a small positive effect
consultation_bonus = (
    (feedback_encounters["consultation_time_minutes"] - 30) / 40
).clip(lower=-0.5, upper=1)

# Random patient-level variation
random_component = np.random.normal(0, 0.75, n_feedback)

experience_score = (
    4.0
    - wait_penalty * 0.45
    + consultation_bonus * 0.20
    + random_component
)

# Convert to 1–5 rating
overall_rating = np.clip(
    np.round(experience_score),
    1,
    5
).astype(int)

# Related ratings with small variation
doctor_rating = np.clip(
    overall_rating + np.random.choice(
        [-1, 0, 0, 0, 1],
        size=n_feedback
    ),
    1,
    5
).astype(int)

wait_time_rating = np.clip(
    6 - np.ceil(
        feedback_encounters["wait_time_minutes"] / 45
    ),
    1,
    5
).astype(int)

service_rating = np.clip(
    overall_rating + np.random.choice(
        [-1, 0, 0, 1],
        size=n_feedback
    ),
    1,
    5
).astype(int)

# Satisfaction score on 0–100 scale
satisfaction_score = (
    overall_rating * 20
    + np.random.normal(0, 4, n_feedback)
).clip(0, 100).round(1)

# Complaints are more likely with low satisfaction
complaint_probability = np.where(
    satisfaction_score < 50,
    0.35,
    np.where(
        satisfaction_score < 70,
        0.12,
        0.03
    )
)

complaint_flag = (
    np.random.random(n_feedback) < complaint_probability
)

# Feedback channel
feedback_channel = np.random.choice(
    ["SMS", "Mobile App", "Email", "Kiosk", "Call Center"],
    size=n_feedback,
    p=[0.35, 0.25, 0.15, 0.15, 0.10]
)

# Feedback date = encounter date
fact_patient_feedback = feedback_encounters[
    [
        "feedback_id",
        "encounter_id",
        "patient_id",
        "doctor_id",
        "department_id",
        "hospital_id",
        "appointment_date_id"
    ]
].copy()

fact_patient_feedback.rename(
    columns={"appointment_date_id": "feedback_date_id"},
    inplace=True
)

fact_patient_feedback["overall_rating"] = overall_rating
fact_patient_feedback["doctor_rating"] = doctor_rating
fact_patient_feedback["wait_time_rating"] = wait_time_rating
fact_patient_feedback["service_rating"] = service_rating
fact_patient_feedback["satisfaction_score"] = satisfaction_score
fact_patient_feedback["complaint_flag"] = complaint_flag
fact_patient_feedback["feedback_channel"] = feedback_channel

print("FACT PATIENT FEEDBACK GENERATED")
print("=" * 60)
print("Rows:", len(fact_patient_feedback))
print("Columns:", len(fact_patient_feedback.columns))

print("\nRating distribution:")
print(fact_patient_feedback["overall_rating"].value_counts().sort_index())

print("\nComplaint rate:")
print(
    round(
        fact_patient_feedback["complaint_flag"].mean() * 100,
        2
    ),
    "%"
)

print("\nMissing values:")
print(fact_patient_feedback.isna().sum())

FACT PATIENT FEEDBACK GENERATED
Rows: 210000
Columns: 14

Rating distribution:
overall_rating
1     1362
2    11562
3    63789
4    95775
5    37512
Name: count, dtype: int64

Complaint rate:
7.75 %

Missing values:
feedback_id           0
encounter_id          0
patient_id            0
doctor_id             0
department_id         0
hospital_id           0
feedback_date_id      0
overall_rating        0
doctor_rating         0
wait_time_rating      0
service_rating        0
satisfaction_score    0
complaint_flag        0
feedback_channel      0
dtype: int64


In [42]:
# FEEDBACK DATA QUALITY VALIDATION

print("FACT PATIENT FEEDBACK VALIDATION")
print("=" * 60)

print("\n1. STRUCTURE")
print("Rows:", len(fact_patient_feedback))
print("Unique feedback IDs:", fact_patient_feedback["feedback_id"].nunique())
print("Unique encounter IDs:", fact_patient_feedback["encounter_id"].nunique())

print("\n2. MISSING VALUES")
print(fact_patient_feedback.isna().sum())

print("\n3. RATING VALIDATION")

rating_cols = [
    "overall_rating",
    "doctor_rating",
    "wait_time_rating",
    "service_rating"
]

for col in rating_cols:
    print(
        f"{col}:",
        "min =", fact_patient_feedback[col].min(),
        "| max =", fact_patient_feedback[col].max(),
        "| invalid =",
        ((fact_patient_feedback[col] < 1) |
         (fact_patient_feedback[col] > 5)).sum()
    )

print("\n4. SATISFACTION SCORE")
print("Min:", fact_patient_feedback["satisfaction_score"].min())
print("Max:", fact_patient_feedback["satisfaction_score"].max())
print(
    "Invalid:",
    ((fact_patient_feedback["satisfaction_score"] < 0) |
     (fact_patient_feedback["satisfaction_score"] > 100)).sum()
)

print("\n5. COMPLAINT FLAG")
print(fact_patient_feedback["complaint_flag"].value_counts())

print("\n6. FEEDBACK CHANNEL")
print(fact_patient_feedback["feedback_channel"].value_counts())

print("\n7. ENCOUNTER LINK VALIDATION")

valid_encounters = set(fact_encounters["encounter_id"])

invalid_encounters = (
    ~fact_patient_feedback["encounter_id"].isin(valid_encounters)
).sum()

print("Invalid encounter IDs:", invalid_encounters)

print("\n" + "=" * 60)
print("FEEDBACK VALIDATION COMPLETE")

FACT PATIENT FEEDBACK VALIDATION

1. STRUCTURE
Rows: 210000
Unique feedback IDs: 210000
Unique encounter IDs: 210000

2. MISSING VALUES
feedback_id           0
encounter_id          0
patient_id            0
doctor_id             0
department_id         0
hospital_id           0
feedback_date_id      0
overall_rating        0
doctor_rating         0
wait_time_rating      0
service_rating        0
satisfaction_score    0
complaint_flag        0
feedback_channel      0
dtype: int64

3. RATING VALIDATION
overall_rating: min = 1 | max = 5 | invalid = 0
doctor_rating: min = 1 | max = 5 | invalid = 0
wait_time_rating: min = 1 | max = 5 | invalid = 0
service_rating: min = 1 | max = 5 | invalid = 0

4. SATISFACTION SCORE
Min: 8.2
Max: 100.0
Invalid: 0

5. COMPLAINT FLAG
complaint_flag
False    193726
True      16274
Name: count, dtype: int64

6. FEEDBACK CHANNEL
feedback_channel
SMS            73121
Mobile App     52691
Email          31547
Kiosk          31439
Call Center    21202
Name: count

In [43]:
# SAVE FACT PATIENT FEEDBACK

feedback_path = r"C:\Users\janak\MediCore_Healthcare_Analytics\data\raw\fact_patient_feedback.csv"

fact_patient_feedback.to_csv(
    feedback_path,
    index=False
)

print("Saved successfully:")
print(feedback_path)
print("Rows:", len(fact_patient_feedback))
print("Columns:", len(fact_patient_feedback.columns))

Saved successfully:
C:\Users\janak\MediCore_Healthcare_Analytics\data\raw\fact_patient_feedback.csv
Rows: 210000
Columns: 14


In [44]:
# FINAL MEDICORE DATASET CHECK

tables = {
    "dim_date": dim_date,
    "dim_hospital": dim_hospital,
    "dim_department": dim_department,
    "dim_doctor": dim_doctor,
    "dim_patient": dim_patient,
    "dim_insurance": dim_insurance,
    "dim_diagnosis": dim_diagnosis,
    "fact_appointments": fact_appointments,
    "fact_encounters": fact_encounters,
    "fact_billing": fact_billing,
    "fact_patient_feedback": fact_patient_feedback
}

print("MEDICORE HEALTHCARE NETWORK")
print("FINAL DATASET CHECK")
print("=" * 65)

for name, df in tables.items():
    print(
        f"{name:25} | "
        f"Rows: {len(df):>7,} | "
        f"Columns: {len(df):>2} | "
        f"Missing cells: {df.isna().sum().sum():>7,}"
    )

print("=" * 65)
print("Total tables:", len(tables))

MEDICORE HEALTHCARE NETWORK
FINAL DATASET CHECK
dim_date                  | Rows:     731 | Columns: 731 | Missing cells:       0
dim_hospital              | Rows:       8 | Columns:  8 | Missing cells:       0
dim_department            | Rows:      44 | Columns: 44 | Missing cells:       0
dim_doctor                | Rows:     600 | Columns: 600 | Missing cells:       0
dim_patient               | Rows: 100,000 | Columns: 100000 | Missing cells:  14,068
dim_insurance             | Rows:       8 | Columns:  8 | Missing cells:       0
dim_diagnosis             | Rows:      12 | Columns: 12 | Missing cells:       0
fact_appointments         | Rows: 500,000 | Columns: 500000 | Missing cells: 533,466
fact_encounters           | Rows: 300,000 | Columns: 300000 | Missing cells:       0
fact_billing              | Rows: 300,000 | Columns: 300000 | Missing cells:  84,800
fact_patient_feedback     | Rows: 210,000 | Columns: 210000 | Missing cells:       0
Total tables: 11


In [45]:
# CORRECTED FINAL MEDICORE DATASET CHECK

print("MEDICORE HEALTHCARE NETWORK")
print("FINAL DATASET CHECK")
print("=" * 75)

for name, df in tables.items():
    print(
        f"{name:25} | "
        f"Rows: {df.shape[0]:>7,} | "
        f"Columns: {df.shape[1]:>2} | "
        f"Missing cells: {df.isna().sum().sum():>7,}"
    )

print("=" * 75)
print("Total tables:", len(tables))

MEDICORE HEALTHCARE NETWORK
FINAL DATASET CHECK
dim_date                  | Rows:     731 | Columns: 11 | Missing cells:       0
dim_hospital              | Rows:       8 | Columns:  7 | Missing cells:       0
dim_department            | Rows:      44 | Columns:  5 | Missing cells:       0
dim_doctor                | Rows:     600 | Columns:  9 | Missing cells:       0
dim_patient               | Rows: 100,000 | Columns: 10 | Missing cells:  14,068
dim_insurance             | Rows:       8 | Columns:  6 | Missing cells:       0
dim_diagnosis             | Rows:      12 | Columns:  4 | Missing cells:       0
fact_appointments         | Rows: 500,000 | Columns: 14 | Missing cells: 533,466
fact_encounters           | Rows: 300,000 | Columns: 18 | Missing cells:       0
fact_billing              | Rows: 300,000 | Columns: 23 | Missing cells:  84,800
fact_patient_feedback     | Rows: 210,000 | Columns: 14 | Missing cells:       0
Total tables: 11


In [46]:
import os

raw_path = r"C:\Users\janak\MediCore_Healthcare_Analytics\data\raw"

expected_files = [
    "dim_date.csv",
    "dim_hospital.csv",
    "dim_department.csv",
    "dim_doctor.csv",
    "dim_patient.csv",
    "dim_insurance.csv",
    "dim_diagnosis.csv",
    "fact_appointments.csv",
    "fact_encounters.csv",
    "fact_billing.csv",
    "fact_patient_feedback.csv"
]

print("MEDICORE RAW DATA FILE CHECK")
print("=" * 65)

for file in expected_files:
    path = os.path.join(raw_path, file)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{file:30} | EXISTS | {size_mb:>8.2f} MB")
    else:
        print(f"{file:30} | MISSING")

print("=" * 65)

MEDICORE RAW DATA FILE CHECK
dim_date.csv                   | EXISTS |     0.05 MB
dim_hospital.csv               | EXISTS |     0.00 MB
dim_department.csv             | EXISTS |     0.00 MB
dim_doctor.csv                 | EXISTS |     0.04 MB
dim_patient.csv                | EXISTS |     7.88 MB
dim_insurance.csv              | EXISTS |     0.00 MB
dim_diagnosis.csv              | EXISTS |     0.00 MB
fact_appointments.csv          | EXISTS |    49.99 MB
fact_encounters.csv            | EXISTS |    56.85 MB
fact_billing.csv               | MISSING
fact_patient_feedback.csv      | EXISTS |    15.66 MB


In [47]:
# SAVE FINAL FACT BILLING

billing_path = r"C:\Users\janak\MediCore_Healthcare_Analytics\data\raw\fact_billing.csv"

fact_billing.to_csv(billing_path, index=False)

print("fact_billing.csv saved successfully")
print("Rows:", len(fact_billing))
print("Columns:", len(fact_billing.columns))
print("Path:", billing_path)

fact_billing.csv saved successfully
Rows: 300000
Columns: 23
Path: C:\Users\janak\MediCore_Healthcare_Analytics\data\raw\fact_billing.csv
